# Auditoria manual DENUE: registros en `revisar`

Este notebook parte de la tabla completa ya clasificada (`denue_textil_distancias.csv`), filtra solo los establecimientos que quedaron como `revisar`, y genera tres tablas de auditoria: Huejotzingo, Santa Ana Xalmimilulco y San Martin Texmelucan.

La idea es revisar cada tabla y llenar `categoria_auditada` con una de estas opciones: `alta`, `media` o `excluir`. El flujo final conserva solo `alta_relevancia_ambiental` y `media_relevancia_ambiental` para producir mapas auditados por localidad.

In [1]:
from pathlib import Path
import subprocess
import sys

import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd()
if ROOT.name.lower() == 'notebooks':
    ROOT = ROOT.parent

TABLES = ROOT / 'outputs' / 'tables'
MAPS = ROOT / 'outputs' / 'maps'
SRC = TABLES / 'denue_textil_distancias.csv'

REGIONS = {
    'huejotzingo': {
        'label': 'Huejotzingo',
        'source_key': 'huejotzingo',
        'audit_file': TABLES / 'auditoria_revisar_huejotzingo.csv',
    },
    'xalmimilulco': {
        'label': 'Santa Ana Xalmimilulco',
        'source_key': 'xalmimilulco',
        'audit_file': TABLES / 'auditoria_revisar_xalmimilulco.csv',
    },
    'san_martin': {
        'label': 'San Martin Texmelucan',
        'source_key': 'san_martin',
        'audit_file': TABLES / 'auditoria_revisar_san_martin.csv',
    },
}

def norm_text(value):
    return str(value).lower().replace('á','a').replace('é','e').replace('í','i').replace('ó','o').replace('ú','u')

def canonical_category(value):
    text = norm_text(value).strip().replace(' ', '_')
    if text in {'alta', 'alto', 'alta_relevancia', 'alta_relevancia_ambiental'}:
        return 'alta_relevancia_ambiental'
    if text in {'media', 'medio', 'mediana', 'media_relevancia', 'media_relevancia_ambiental'}:
        return 'media_relevancia_ambiental'
    if text in {'excluir', 'no', 'no_pertinente', 'descartar'}:
        return 'excluir'
    if text in {'', 'nan', 'none'}:
        return ''
    return text

display(Markdown(f'Proyecto: `{ROOT}`'))

Proyecto: `c:\Users\narro\OneDrive - Fundacion Universidad de las Americas Puebla\servicio social atoyac`

## 1. Leer tabla completa y filtrar `revisar`

In [2]:
denue = pd.read_csv(SRC, encoding='utf-8-sig')
revisar = denue[denue['categoria_relevancia_ambiental'].astype(str).eq('revisar')].copy()

display(Markdown(f'### Tabla completa: **{len(denue):,}** registros'))
display(Markdown(f'### Registros en revision: **{len(revisar):,}**'))
display(revisar['source_folder'].value_counts(dropna=False).rename_axis('source_folder').reset_index(name='registros'))
display(revisar.head(10))

### Tabla completa: **1,067** registros

### Registros en revision: **182**

,source_folder,registros
0,agebs\san_martin,101
1,agebs\xalmimilulco,52
2,agebs\huejotzingo,29


,id,clee,nombre_de_la_unidad_economica,codigo_de_la_clase_actividad_scian,latitud,longitud,source_path,source_file,source_folder,layer_type,...,codigo_scian_detectado,palabras_clave_detectadas,categoria_relevancia_ambiental,nota_metodologica,distancia_hidrografia_m,en_buffer_100m,en_buffer_250m,en_buffer_500m,en_buffer_1000m,rango_distancia_hidrografia
1,3325813,21074313210000056111000000U9,INDUSTRIAS MAQUIN,313230,19.161344,-98.391562,data\raw\agebs\huejotzingo\INEGI_DENUE_.shp,INEGI_DENUE_.shp,agebs\huejotzingo,denue,...,313230,NaN,revisar,"Priorizacion preliminar por produccion, confec...",921.240971,False,False,False,True,500-1000 m
2,3325963,21074315225000111000000000U8,SONIA MENDEZ NOVIAS,315225,19.163965,-98.404081,data\raw\agebs\huejotzingo\INEGI_DENUE_.shp,INEGI_DENUE_.shp,agebs\huejotzingo,denue,...,315225,NaN,revisar,"Priorizacion preliminar por produccion, confec...",1041.667327,False,False,False,False,>1000 m
8,8271641,21074465915000021010000000U3,CREACIONES MAYRA,315224,19.165291,-98.406506,data\raw\agebs\huejotzingo\INEGI_DENUE_.shp,INEGI_DENUE_.shp,agebs\huejotzingo,denue,...,315224,NaN,revisar,"Priorizacion preliminar por produccion, confec...",363.020287,False,False,True,True,250-500 m
19,10118182,21074812210000811000000000U3,CLEAN CLOTHES,812210,19.148389,-98.405906,data\raw\agebs\huejotzingo\INEGI_DENUE_.shp,INEGI_DENUE_.shp,agebs\huejotzingo,denue,...,812210,NaN,revisar,"Priorizacion preliminar por produccion, confec...",604.944739,False,False,False,True,500-1000 m
21,10205167,21074313210000083000000000U3,BODEGA PUNTO FINO EN TELAS,313210,19.162693,-98.386820,data\raw\agebs\huejotzingo\INEGI_DENUE_.shp,INEGI_DENUE_.shp,agebs\huejotzingo,denue,...,313210,NaN,revisar,"Priorizacion preliminar por produccion, confec...",1062.523188,False,False,False,False,>1000 m
23,10340864,21074812210000731000000000U3,TINTOWASH,812210,19.161238,-98.408648,data\raw\agebs\huejotzingo\INEGI_DENUE_.shp,INEGI_DENUE_.shp,agebs\huejotzingo,denue,...,812210,NaN,revisar,"Priorizacion preliminar por produccion, confec...",1184.036585,False,False,False,False,>1000 m
47,6378222,21074315191000014013000000U8,MARTINTER,315122,19.143896,-98.386132,data\raw\agebs\huejotzingo\INEGI_DENUE_.shp,INEGI_DENUE_.shp,agebs\huejotzingo,denue,...,315122,NaN,revisar,"Priorizacion preliminar por produccion, confec...",1139.931290,False,False,False,False,>1000 m
48,6380168,21074313113000045001000000U8,VERTICAL SPIN,313112,19.157825,-98.385117,data\raw\agebs\huejotzingo\INEGI_DENUE_.shp,INEGI_DENUE_.shp,agebs\huejotzingo,denue,...,313112,NaN,revisar,"Priorizacion preliminar por produccion, confec...",339.020753,False,False,True,True,250-500 m
49,6380169,21074313113000035010000000U9,TRITON INDUSTRIAL,313112,19.160888,-98.386443,data\raw\agebs\huejotzingo\INEGI_DENUE_.shp,INEGI_DENUE_.shp,agebs\huejotzingo,denue,...,313112,NaN,revisar,"Priorizacion preliminar por produccion, confec...",101.191190,False,True,True,True,100-250 m
50,6856149,21074315191000026011000000U2,SEAMLESS GLOBAL SOLUTIONS,315122,19.162631,-98.386641,data\raw\agebs\huejotzingo\INEGI_DENUE_.shp,INEGI_DENUE_.shp,agebs\huejotzingo,denue,...,315122,NaN,revisar,"Priorizacion preliminar por produccion, confec...",1232.965549,False,False,False,False,>1000 m


## 2. Crear plantillas por localidad

Estas plantillas se crean solo si no existen, para no borrar tus cambios. Puedes editarlas desde Excel/LibreOffice o cargar/modificar las variables en el notebook. Llena `categoria_auditada` con `alta`, `media` o `excluir`, y usa `notas_auditoria` para justificar dudas.

In [3]:
AUDIT_COLUMNS = [
    'id', 'clee', 'nombre_de_la_unidad_economica', 'codigo_de_la_clase_actividad_scian',
    'source_folder', 'categoria_relevancia_ambiental', 'palabras_clave_detectadas',
    'distancia_hidrografia_m', 'rango_distancia_hidrografia', 'latitud', 'longitud',
    'categoria_auditada', 'notas_auditoria'
]

templates = {}
for region_id, meta in REGIONS.items():
    mask = revisar['source_folder'].map(norm_text).str.contains(meta['source_key'], na=False)
    region_df = revisar.loc[mask].copy()
    region_df['categoria_auditada'] = ''
    region_df['notas_auditoria'] = ''
    region_df = region_df[[c for c in AUDIT_COLUMNS if c in region_df.columns]]
    path = meta['audit_file']
    if not path.exists():
        region_df.to_csv(path, index=False, encoding='utf-8-sig')
        print(f'Creada: {path.relative_to(ROOT)} ({len(region_df)} filas)')
    else:
        print(f'Ya existe, no se sobreescribe: {path.relative_to(ROOT)}')
    templates[region_id] = pd.read_csv(path, encoding='utf-8-sig')

display(Markdown('### Plantillas cargadas'))
display(pd.DataFrame([
    {'localidad': meta['label'], 'archivo': str(meta['audit_file'].relative_to(ROOT)), 'filas': len(templates[region_id])}
    for region_id, meta in REGIONS.items()
]))

Ya existe, no se sobreescribe: outputs\tables\auditoria_revisar_huejotzingo.csv
Ya existe, no se sobreescribe: outputs\tables\auditoria_revisar_xalmimilulco.csv
Ya existe, no se sobreescribe: outputs\tables\auditoria_revisar_san_martin.csv


### Plantillas cargadas

,localidad,archivo,filas
0,Huejotzingo,outputs\tables\auditoria_revisar_huejotzingo.csv,29
1,Santa Ana Xalmimilulco,outputs\tables\auditoria_revisar_xalmimilulco.csv,52
2,San Martin Texmelucan,outputs\tables\auditoria_revisar_san_martin.csv,101


## 3. Tablas para auditoria

In [4]:
SHOW_COLUMNS = [
    'id', 'nombre_de_la_unidad_economica', 'codigo_de_la_clase_actividad_scian',
    'palabras_clave_detectadas', 'distancia_hidrografia_m', 'rango_distancia_hidrografia',
    'categoria_auditada', 'notas_auditoria'
]

for region_id, meta in REGIONS.items():
    df = templates[region_id]
    display(Markdown(f"## {meta['label']} - registros por auditar: **{len(df):,}**"))
    display(df[[c for c in SHOW_COLUMNS if c in df.columns]])

## Huejotzingo - registros por auditar: **29**

,id,nombre_de_la_unidad_economica,codigo_de_la_clase_actividad_scian,palabras_clave_detectadas,distancia_hidrografia_m,rango_distancia_hidrografia,categoria_auditada,notas_auditoria
0,3325813,INDUSTRIAS MAQUIN,313230,NaN,921.240971,500-1000 m,NaN,NaN
1,3325963,SONIA MENDEZ NOVIAS,315225,NaN,1041.667327,>1000 m,NaN,NaN
2,8271641,CREACIONES MAYRA,315224,NaN,363.020287,250-500 m,NaN,NaN
3,10118182,CLEAN CLOTHES,812210,NaN,604.944739,500-1000 m,NaN,NaN
4,10205167,BODEGA PUNTO FINO EN TELAS,313210,NaN,1062.523188,>1000 m,NaN,NaN
5,10340864,TINTOWASH,812210,NaN,1184.036585,>1000 m,NaN,NaN
6,6378222,MARTINTER,315122,NaN,1139.931290,>1000 m,NaN,NaN
7,6380168,VERTICAL SPIN,313112,NaN,339.020753,250-500 m,NaN,NaN
8,6380169,TRITON INDUSTRIAL,313112,NaN,101.191190,100-250 m,NaN,NaN
9,6856149,SEAMLESS GLOBAL SOLUTIONS,315122,NaN,1232.965549,>1000 m,NaN,NaN


## Santa Ana Xalmimilulco - registros por auditar: **52**

,id,nombre_de_la_unidad_economica,codigo_de_la_clase_actividad_scian,palabras_clave_detectadas,distancia_hidrografia_m,rango_distancia_hidrografia,categoria_auditada,notas_auditoria
0,3326133,DESEBRADO DE BLUSA SIN NOMBRE,315229,NaN,318.912769,250-500 m,NaN,NaN
1,3427871,PEGADO DE PRESILLA SIN NOMBRE,315123,NaN,603.960226,500-1000 m,NaN,NaN
2,3428329,POLIPROCESO,313310,NaN,1172.296692,>1000 m,NaN,NaN
3,3538483,HERA APPAREL,315229,NaN,114.449132,100-250 m,NaN,NaN
4,6378259,AUNDE MEXICO,313320,NaN,1060.862493,>1000 m,NaN,NaN
5,6379033,ESPINTEX,313112,NaN,593.496691,500-1000 m,NaN,NaN
6,6379890,SKYTEX MEXICO,313210,NaN,45.219590,0-100 m,NaN,NaN
7,8089248,TERMINADO,315229,NaN,964.284501,500-1000 m,NaN,NaN
8,8089451,DESEBRADORA SIN NOMBRE,315229,NaN,771.290513,500-1000 m,NaN,NaN
9,9650425,MAQUILLA,315229,NaN,631.591903,500-1000 m,NaN,NaN


## San Martin Texmelucan - registros por auditar: **101**

,id,nombre_de_la_unidad_economica,codigo_de_la_clase_actividad_scian,palabras_clave_detectadas,distancia_hidrografia_m,rango_distancia_hidrografia,categoria_auditada,notas_auditoria
0,6271697,FABRICA DE SAN MARTIN,313210,NaN,395.413976,250-500 m,NaN,NaN
1,7536246,LONAS Y LUMINOSOS KENIA,314912,NaN,359.403053,250-500 m,NaN,NaN
2,8787568,COSTUREROS SIN NOMBRE,315229,NaN,711.872836,500-1000 m,NaN,NaN
3,9176687,COMPOSTURAS DE ROPA SIN NOMBRE,315223,NaN,150.220521,100-250 m,NaN,NaN
4,9329155,PLANCHADURIA,315229,NaN,222.776208,100-250 m,NaN,NaN
...,...,...,...,...,...,...,...,...
96,11464962,COSTURERIA SIN NOMBRE,315225,NaN,417.546387,250-500 m,NaN,NaN
97,11663310,COSTURERIA SIN NOMBRE,315225,NaN,291.317042,250-500 m,NaN,NaN
98,11816167,PLANCHADURIA SIN NOMBRE,812210,NaN,98.375109,0-100 m,NaN,NaN
99,12317062,LAVARTEX TLAXCALA,812210,NaN,128.346440,100-250 m,NaN,NaN


## 4. Opcional: auditar desde el notebook con diccionarios

Si prefieres editar aqui en vez de editar los CSV, escribe decisiones por `id`. Ejemplo: `3325813: ('media', 'SCIAN 313; revisar como productivo')`. Luego corre la celda para actualizar las plantillas.

In [ ]:
decisiones_huejotzingo = {
    3325813: ('alta', 'Fabricantes y maquiladores especializados en telas no tejidas y productos higiénicos desechables.'),
    3325963: ('alta', 'Taller de alta costura enfocado en el diseño y confección a medida de vestidos de novia, XV años y gala.'),
    8271641: ('alta', 'Taller artesanal dedicado al diseño y confección de trajes típicos y gaznes bordados para el Carnaval de Huejotzingo.'),
    10118182: ('media', 'Establecimiento encargado del servicio comercial de lavandería por kilo, secado y planchado de ropa de uso diario.'),
    10205167: ('alta', 'Bodega mayorista de insumos textiles especializada en la fabricación y venta de tejidos de punto de alta calidad.'),
    10340864: ('media', 'Lavandería y tintorería industrial enfocada en el lavado, teñido y tratamiento especial de acabados textiles.'),
    6378222: ('alta', 'Empresa de servicios de tintura, acabado y maquila industrial para la cadena productiva del sector textil.'),
    6380168: ('alta', 'Planta industrial textil dedicada a la hilatura, tejido y acabados de fibras sintéticas y naturales.'),
    6380169: ('alta', 'Fabricante especializado en empaques, contenedores industriales o insumos plásticos de alta resistencia para logística.'),
    6856149: ('alta', 'Compañía manufacturera enfocada en la producción textil de prendas sin costuras mediante tecnología seamless.'),
    6874492: ('alta', 'Fábrica textil orientada a la producción, tejido y comercialización al por mayor de hilos y telas industriales.'),
    6874618: ('alta', 'Empresa dedicada a la manufactura, acabado y distribución de productos textiles y telas para confección.'),
    6875485: ('alta', 'Consorcio textil especializado en la comercialización, distribución e importación de hilos y telas a gran escala.'),
    8271868: ('alta', 'Taller tradicional de diseño, confección y renta de indumentaria de gala para los batallones del Carnaval.'),
    8272057: ('alta', 'Establecimiento artesanal dedicado a la elaboración y bordado manual de trajes típicos para danzantes del Carnaval.'),
    8559701: ('alta', 'Planta textil enfocada en el procesamiento, hilatura y reciclaje de fibras de poliéster y algodón.'),
    9440701: ('alta', 'Fábrica de componentes y aislantes acústicos/térmicos, principalmente para la industria automotriz y de la construcción.'),
    9522648: ('media', 'Centro de servicios de lavandería, desarrugado y procesos de terminación para la industria de la confección local.'),
    9567689: ('alta', 'Local comercial y taller de costura especializado en el diseño y confección de vestidos de bodas y ceremonias.'),
    9844272: ('alta', 'Taller de manufactura textil orientado a la producción de prendas de vestir y maquila de ropa casual.'),
    10066057: ('alta', 'Taller de bordado industrial y computarizado para uniformes, logotipos empresariales y prendas personalizadas.'),
    10582863: ('alta', 'Fabricante de insumos de mercería especializado en cintas tejidas, etiquetas textiles y elásticos para confección.'),
    10597677: ('alta', 'Planta transnacional líder en la fabricación de telas y cubiertas tejidas para colchones y la industria del descanso.'),
    10978442: ('media', 'Proveedor de servicios de lavandería industrial, desinfectado y procesos especiales de lavado para empresas.'),
    11095597: ('alta', 'Taller artesanal de escultura y pintura especializado en la elaboración de máscaras tradicionales de madera y piel para el Carnaval.'),
    11484735: ('media', 'Establecimiento comercial dedicado a la venta de ropa, calzado o accesorios de moda para el mercado local.'),
    11488206: ('alta', 'Fábrica textil de tejido de punto dedicada a la producción de calcetería, ropa interior y prendas básicas.'),
    11620168: ('alta', 'Negocio de artesanos locales dedicados a la creación de artesanías, accesorios y adornos alegóricos para fiestas patronales.'),
    11827938: ('alta', 'Multinacional petroquímica que produce filamentos y fibras de poliéster de alto rendimiento para bolsas de aire y neumáticos.')
}


decisiones_xalmimilulco = {
    3326133: ('baja', 'Taller local especializado en el deshebrado, limpieza de hilos y terminado de blusas confeccionadas.'),
    3427871: ('baja', 'Pequeño taller de maquila enfocado en el pegado de presillas y costuras de refuerzo en pantalones y prendas.'),
    3428329: ('alta', 'Empresa de procesos industriales dedicada al teñido, lavado especial y acabado de telas y prendas de mezclilla.'),
    3538483: ('alta', 'Planta manufacturera de confección textil a gran escala que produce prendas de vestir para marcas nacionales e internacionales.'),
    6378259: ('alta', 'Fábrica automotriz transnacional especializada en el diseño y producción de textiles técnicos y fundas para asientos de vehículos.'),
    6379033: ('alta', 'Hilandería industrial enfocada en la producción y distribución al por mayor de hilo de algodón 100% peinado de alta calidad.'),
    6379890: ('alta', 'Complejo textil líder en la fabricación, tejido, teñido y acabado de telas de poliéster, nylon y fieltros estructurados.'),
    8089248: ('baja', 'Taller secundario dedicado al proceso de terminado, planchado y empaque final de piezas de ropa.'),
    8089451: ('baja', 'Microempresa de costura orientada a la limpieza de hilos sueltos (deshebrado) en lotes de confección local.'),
    9650425: ('media', 'Establecimiento de costura y ensamble que trabaja bajo el modelo de maquila para marcas de ropa.'),
    9730300: ('baja', 'Taller dedicado a la revisión de calidad, desmanchado y procesos finales (terminado) de prendas textiles.'),
    9800677: ('baja', 'Taller especializado en la costura de marquillas, etiquetas de marca y tallas en ropa terminada.'),
    9804661: ('media', 'Negocio local de maquila textil dedicado al ensamble de piezas cortadas para pantalones, camisas o sudaderas.'),
    9811409: ('baja', 'Taller de costura familiar registrado bajo nombre propio, dedicado a la confección y maquila a pequeña escala.'),
    9877804: ('media', 'Taller de confección general encargado del ensamble de ropa por contrato para distribuidores mayoristas.'),
    9920113: ('baja', 'Establecimiento enfocado en el planchado industrial, desarrugado por vapor y empaque en bolsas de lotes de ropa.'),
    10029125: ('media', 'Taller de costura externo enfocado en la producción intermedia y habilitación de prendas de vestir.'),
    10105987: ('baja', 'Taller de confección y costura operado de manera independiente por diseño o maquila bajo pedido.'),
    10401755: ('alta', 'Fábrica o comercializadora textil enfocada en procesos de lavado, teñido de mezclilla o manufactura de ropa casual.'),
    10516759: ('media', 'Taller de confección dedicado a la costura y armado de piezas textiles bajo el formato de maquila.'),
    10541104: ('media', 'Establecimiento de producción textil enfocado en el ensamble a destajo de prendas de vestir.'),
    10623693: ('baja', 'Taller complementario de costura especializado exclusivamente en colocar presillas para cinturones y remaches textiles.'),
    10687795: ('media', 'Negocio de confección textil que ofrece servicios de costura recta y overlock para excedentes de producción.'),
    10752985: ('baja', 'Espacio técnico enfocado en la preparación, empaquetado y control de calidad final antes de la distribución de ropa.'),
    10815242: ('media', 'Línea de maquila local encargada de la costura y unión de piezas textiles cortadas.'),
    11081907: ('media', 'Bodega de almacenamiento de materia prima, rollos de tela o hilos para el abastecimiento de talleres cercanos.'),
    11084020: ('media', 'Taller de costura mediano dedicado a la manufactura y armado de prendas para el mercado mayorista.'),
    11099113: ('media', 'Taller industrial de planchado y transferencia de calor (estampado/vinil) en prendas terminadas.'),
    11155463: ('media', 'Centro de acopio y almacenamiento de producto terminado listo para su distribución a los mercados de ropa regionales.'),
    11183668: ('baja', 'Bodega de reciclaje y comercialización de desperdicio textil, retacería y remanentes de tela.'),
    11221708: ('baja', 'Taller de costura especializado en el pegado, habilitación y confección de bolsas traseras para pantalones y jeans.'),
    11313187: ('media', 'Taller de maquila y confección general enfocado en el ensamble y armado de piezas textiles bajo pedido.'),
    11332144: ('baja', 'Taller técnico complementario dedicado de manera exclusiva a costuras de remate con máquina overlock.'),
    11382766: ('baja', 'Taller familiar de costura y confección independiente registrado a nombre de su propietario.'),
    11467703: ('media', 'Establecimiento dedicado a la maquila de ropa, ensamble de prendas y maquila de costura para mayoristas.'),
    3326283: ('baja', 'Taller de alta costura, sastrería artesanal y diseño de prendas de vestir personalizadas a la medida.'),
    6858807: ('alta', 'Planta industrial dedicada a la fabricación de fieltros punzonados, guatas y textiles técnicos no tejidos.'),
    6861846: ('alta', 'Lavandería industrial y tintorería especializada en procesos de deslave, piedra (stone wash) y acabados para mezclilla.'),
    8692099: ('media', 'Taller de impresión publicitaria, serigrafía y bordado enfocado en artículos promocionales y uniformes textiles.'),
    9706371: ('baja', 'Negocio local de costura y confección textil operado de manera independiente por su propietario.'),
    9747796: ('media', 'Taller de manufactura especializado en grabado y confección de etiquetas de piel y parches traseros para pantalones.'),
    10122074: ('media', 'Línea de producción textil dedicada al ensamble, confección y habilitación de prendas de vestir.'),
    10122585: ('baja', 'Pequeño taller de costura y acabados textiles registrado bajo el nombre del productor local.'),
    10262715: ('baja', 'Taller de impresión y serigrafía enfocado en la personalización de servilletas de tela y recuerdos para eventos sociales.'),
    10328922: ('media', 'Taller de confección y costura que ofrece servicios de maquila a destajo para marcas de ropa regionales.'),
    10416578: ('media', 'Establecimiento de producción textil dedicado al ensamble intermedio de piezas y costura recta.'),
    10451408: ('alta', 'Planta de procesos textiles (sucursal/área de procesos) enfocada en el teñido, suavizado y terminado de jeans.'),
    10500192: ('media', 'Taller de costura general encargado del armado y unión de lotes de ropa cortada.'),
    10724349: ('media', 'Taller o bodega textil dedicada a la distribución, corte o confección de prendas y uniformes industriales.'),
    10939082: ('baja', 'Pequeño taller familiar enfocado en el diseño, costura y confección de ropa casual o infantil.'),
    11198278: ('baja', 'Taller especializado en el proceso de pretinado, fijado y armado de pretinas para pantalones de mezclilla.'),
    11521781: ('media', 'Taller de maquila textil dedicado al ensamble general de ropa y costura bajo contrato.')
}




decisiones_san_martin = {
    # id: ('alta' o 'media' o 'excluir', 'nota'),
}

DECISIONES = {
    'huejotzingo': decisiones_huejotzingo,
    'xalmimilulco': decisiones_xalmimilulco,
    'san_martin': decisiones_san_martin,
}

for region_id, decisions in DECISIONES.items():
    if not decisions:
        continue
    path = REGIONS[region_id]['audit_file']
    df = pd.read_csv(path, encoding='utf-8-sig')
    for denue_id, value in decisions.items():
        if isinstance(value, tuple):
            category, note = value
        else:
            category, note = value, ''
        mask = df['id'].astype(str).eq(str(denue_id))
        df.loc[mask, 'categoria_auditada'] = category
        df.loc[mask, 'notas_auditoria'] = note
    df.to_csv(path, index=False, encoding='utf-8-sig')
    print(f'Actualizada: {path.relative_to(ROOT)}')

## 5. Consolidar auditoria

Corre esta parte despues de editar las tres plantillas. Esto pega las tres tablas en una sola salida de revision manual.

In [ ]:
audited_parts = []
for region_id, meta in REGIONS.items():
    df = pd.read_csv(meta['audit_file'], encoding='utf-8-sig')
    df['region_auditoria'] = region_id
    df['categoria_auditada_normalizada'] = df.get('categoria_auditada', '').map(canonical_category)
    audited_parts.append(df)

auditada = pd.concat(audited_parts, ignore_index=True)
out = TABLES / 'denue_revisar_categorias_auditadas.csv'
auditada.to_csv(out, index=False, encoding='utf-8-sig')

display(Markdown(f'### Tabla auditada guardada: `{out.relative_to(ROOT)}`'))
display(auditada['categoria_auditada_normalizada'].replace('', 'sin_auditar').value_counts().rename_axis('categoria').reset_index(name='registros'))
display(auditada.head(20))

## 6. Aplicar auditoria y generar mapas finales

Este paso genera `denue_categorias_auditadas.csv`, `denue_textil_auditado.gpkg` y los mapas `14_*_denue_textil_auditado_alta_media.png`, uno por localidad. No borra ni reemplaza los mapas anteriores con puntos `revisar`; crea una version posterior ya auditada, solo con `alta` y `media`.

In [ ]:
subprocess.run([sys.executable, str(ROOT / 'scripts' / '07_apply_denue_audit.py')], cwd=ROOT, check=True)
subprocess.run([sys.executable, str(ROOT / 'scripts' / '08_make_audited_maps.py')], cwd=ROOT, check=True)

display(Markdown('### Outputs principales'))
for path in [
    TABLES / 'denue_categorias_auditadas.csv',
    TABLES / 'resumen_denue_categorias_auditadas_por_localidad.csv',
    MAPS / '14_huejotzingo_denue_textil_auditado_alta_media.png',
    MAPS / '14_xalmimilulco_denue_textil_auditado_alta_media.png',
    MAPS / '14_san_martin_denue_textil_auditado_alta_media.png',
]:
    display(Markdown(f'- `{path.relative_to(ROOT)}`'))